# 기사용 차트 문법 — 같은 데이터, 다른 인상
정직한 차트 6가지 규칙을 코드로 확인해요.

In [ ]:
# ▶ 환경 준비 — Colab · VS Code 공통, 맨 처음 한 번 실행
import os, sys, platform, subprocess
IN_COLAB = 'google.colab' in sys.modules

def gpu_info():
    try:
        r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                           capture_output=True, text=True, timeout=10)
        return r.stdout.strip() or None
    except Exception:
        return None

GPU = gpu_info()
print('실행 위치:', 'Colab' if IN_COLAB else f'내 컴퓨터 ({platform.system()})')
print('GPU:', GPU or '없음 → CPU로 실행 (이 노트북은 CPU로도 충분해요)')

# (선택) pandas를 GPU로 — Colab GPU 런타임 또는 Linux/WSL에 cuDF가 있을 때만
USE_GPU_PANDAS = False          # True로 바꾸고 이 칸을 다시 실행
if USE_GPU_PANDAS and GPU:
    try:
        get_ipython().run_line_magic('load_ext', 'cudf.pandas')
        print('pandas → GPU(cuDF) 사용')
    except Exception as e:
        print('cuDF를 쓸 수 없어 일반 pandas로 실행:', type(e).__name__)

# 한글 글꼴: Colab은 나눔고딕 설치, 윈도우는 맑은 고딕, 맥은 애플고딕
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
if IN_COLAB:
    nanum = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    if not os.path.exists(nanum):
        subprocess.run('apt-get -qq install -y fonts-nanum > /dev/null', shell=True)
    fm.fontManager.addfont(nanum)
    plt.rcParams['font.family'] = 'NanumGothic'
elif platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:                                   # 리눅스·WSL: 설치된 한글 글꼴 찾기
    ko = [f.name for f in fm.fontManager.ttflist if 'Nanum' in f.name or 'CJK KR' in f.name]
    if ko: plt.rcParams['font.family'] = ko[0]
plt.rcParams['axes.unicode_minus'] = False
print('한글 글꼴:', plt.rcParams['font.family'])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
tit = sns.load_dataset('titanic')
mpg = sns.load_dataset('mpg')
flights = sns.load_dataset('flights')

## 규칙 1. 막대는 0부터

In [ ]:
m = mpg.groupby('origin')['mpg'].mean().reindex(['usa', 'europe', 'japan'])
fig, (a, b) = plt.subplots(1, 2, figsize=(12, 4))
a.bar(['미국', '유럽', '일본'], m.values); a.set_ylim(19, 31); a.set_title('세로축 19부터 — 과장')
b.bar(['미국', '유럽', '일본'], m.values); b.set_ylim(0);      b.set_title('세로축 0부터 — 정직')
plt.show()
print(m.round(1))

## 규칙 2. 축은 하나 — 두 값이면 위아래 두 칸 (`sharex=True`)

In [ ]:
yearly = flights.groupby('year')['passengers'].sum()
growth = yearly.pct_change() * 100
fig, (a, b) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
a.bar(yearly.index, yearly.values); a.set_ylabel('연간 승객(천 명)')
b.plot(growth.index, growth.values, marker='o'); b.set_ylabel('전년 대비(%)')
plt.show()

## 규칙 3. 강조는 하나만

In [ ]:
monthly = flights.groupby('month', observed=True)['passengers'].mean()
colors = ['#E8590C' if m in (7, 8) else '#B8B2A7' for m in range(1, 13)]    # 강조할 달만 주황
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar([f'{m}월' for m in range(1, 13)], monthly.values, color=colors)
ax.set_ylabel('평균 승객(천 명)')
plt.show()

## 규칙 4·5. 제목은 기사 문장, 출처·기간·단위는 아래에

In [ ]:
rate = tit.groupby('pclass')['survived'].mean() * 100
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(['1등석', '2등석', '3등석'], rate.values, color=['#E8590C', '#B8B2A7', '#B8B2A7'])
ax.set_ylim(0, 100)
ax.set_title('1등석 생존율, 3등석의 2.6배', loc='left', fontsize=16, fontweight='bold')
fig.text(0.01, 0.01, '출처: 타이타닉 승객 데이터(891명) · 단위: %', fontsize=9, color='gray')
plt.show()

## 실습: 내 차트 3장 고쳐 보기
| 차트 | 어긋난 규칙 | 고친 제목(기사 문장) |
|---|---|---|
| 내 차트 1 | | |
| 내 차트 2 | | |
| 내 차트 3 | | |

In [ ]:
# 고친 차트 코드를 여기에